# Chains in LangChain

1. Sequential Chains 

In [56]:
from langchain_mistralai import ChatMistralAI 
from langchain_core.prompts import PromptTemplate 
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
from pydantic import BaseModel,Field
from typing import List


from dotenv import load_dotenv

Loading api keys and creating model instance

In [61]:
if load_dotenv():
    print("Api keys succesfully loaded")

model = ChatMistralAI(
    model = "mistral-medium-2508",
    temperature = 0.8,
    # max_tokens = 250,
)

print("model successfully built")

Api keys succesfully loaded
model successfully built


In [57]:
class AnimeItem(BaseModel):
    Anime: str = Field(description="name of the anime")
    Summary: str = Field(description="1-2 line short summary")

class OutputFormat(BaseModel):
    animes: List[AnimeItem]

In [58]:
Str_parser = StrOutputParser()
pydantic_parser = PydanticOutputParser(pydantic_object=OutputFormat) 
format_instruction = pydantic_parser.get_format_instructions()

In [59]:
template = PromptTemplate.from_template(
"""
You are an anime expert.

Based on user's interest: {interest}

Recommend 5 anime and generate short summaries.

{format_instruction}
"""
)


In [ ]:
Anime_chain = template | model | pydantic_parser

response = Anime_chain.invoke({
    "interest":"Action",
    "format_instruction" : format_instruction
})

animes=[AnimeItem(Anime='Attack on Titan (Shingeki no Kyojin)', Summary='Humanity fights for survival against giant humanoid monsters called Titans in a world enclosed by massive walls. Eren Yeager vows to eradicate the Titans after his hometown is destroyed, uncovering dark secrets about their origins.'), AnimeItem(Anime='Demon Slayer: Kimetsu no Yaiba', Summary='After his family is slaughtered by demons, Tanjiro Kamado becomes a Demon Slayer to avenge them and find a cure for his sister Nezuko, who was turned into a demon. The anime blends breathtaking action with emotional storytelling.'), AnimeItem(Anime='My Hero Academia (Boku no Hero Academia)', Summary='In a world where superpowers (Quirks) are common, Izuku Midoriya dreams of becoming the greatest hero. After inheriting the power of All Might, he enrolls in U.A. High School to train and face villains threatening society.'), AnimeItem(Anime='One Punch Man', Summary='Saitama, a hero who can defeat any enemy with a single punch, s

In [73]:
for anime in response.animes:
    print(anime.Anime)
    print(anime.Summary)

Attack on Titan (Shingeki no Kyojin)
Humanity fights for survival against giant humanoid monsters called Titans in a world enclosed by massive walls. Eren Yeager vows to eradicate the Titans after his hometown is destroyed, uncovering dark secrets about their origins.
Demon Slayer: Kimetsu no Yaiba
After his family is slaughtered by demons, Tanjiro Kamado becomes a Demon Slayer to avenge them and find a cure for his sister Nezuko, who was turned into a demon. The anime blends breathtaking action with emotional storytelling.
My Hero Academia (Boku no Hero Academia)
In a world where superpowers (Quirks) are common, Izuku Midoriya dreams of becoming the greatest hero. After inheriting the power of All Might, he enrolls in U.A. High School to train and face villains threatening society.
One Punch Man
Saitama, a hero who can defeat any enemy with a single punch, seeks a worthy challenge in a world overrun by monsters and villains. The anime parodies classic superhero tropes with over-the-to

# Parallel Chains 

In [7]:
from langchain_mistralai import ChatMistralAI
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate 
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [3]:
if load_dotenv():
    print("API's loaded successfully.")

model_tech_stack = ChatMistralAI(
    model = "mistral-medium-2508",
    temperature = 0.9,
)

model_code = ChatMistralAI(
    model = "mistral-small-2506",
    temperature = 0.3,
)
print("Model built Successfully")

API's loaded successfully.
Model built Successfully


Creating a small coding chat model that helps with implemenation and also helps you get the tech stack of the given problem

In [ ]:
prompt_1 = PromptTemplate.from_template(
    template = """ You are an expert system designer, your task to provide a feasible system design for {project_name}""",
    input_variable = ['project_name']
)

prompt_2 = PromptTemplate.from_template(
    template = """You are an experienced coder you can handle all languages and write scalable and efficient code. Yout task is to write code for {project_name}""",
    input_variable = ['project_name']
)

prompt_3 = PromptTemplate.from_template(
    template = """You are an expert system designer, your task is to provide tech stack for using {code_implemenation} and {system_design}""",
    input_variables  = ['code_implemenation','system_design']
    
)

In [23]:
parser = StrOutputParser()

In [24]:
# chain-1 will generate a guide on step-by-step implementation plan.
chain_1 = prompt_1 | model_tech_stack | parser 

# chain-2 will generate code for the system. 
chain_2 = prompt_2 | model_code | parser

In [ ]:
# creating a Runnable object that will run the chains Parallelly
parallel_chain = RunnableParallel(
    {
        "system_design" : chain_1,
        "code_implemenation": chain_2
    }
)

In [26]:
merge_chains = prompt_3 | model_tech_stack | parser 
chain = parallel_chain | merge_chains 
response = chain.invoke({
    "project_name" : "Design a student attendance portal."
})

In [36]:
print(response)

# **Enhanced Student Attendance Portal - Tech Stack & Implementation Guide**

Based on your existing design, I'll provide a **comprehensive tech stack** with optimizations, best practices, and deployment strategies to build a **scalable, secure, and high-performance** Student Attendance Portal.

---

## **1. Tech Stack Recommendations**
### **Frontend**
| **Component**       | **Technology** | **Why?** |
|----------------------|---------------|----------|
| **Web Framework**    | Next.js (React) | SSR/SSG for SEO, API routes, better performance |
| **State Management** | Zustand / Redux Toolkit | Lightweight (Zustand) or structured (Redux) |
| **UI Components**   | MUI (Material-UI) / Chakra UI | Pre-built accessible components |
| **Forms**           | React Hook Form + Zod | Validation, performance, DX |
| **Charts**          | Chart.js / Recharts | Interactive attendance analytics |
| **Real-time Updates** | Socket.io | Live attendance tracking |
| **Mobile App**      | React Native

Finally this is how chain looks like \
 ![alt text](chain_graph.png)

# Conditional Chains

In [126]:
from langchain_core.runnables import RunnableBranch ,RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser 
from pydantic import Field,BaseModel
from typing import Literal 

In [120]:
class output_format(BaseModel):
    category : Literal['Action','Comedy','Suspence'] = Field(description="return the category of feedback.")

pydantic_parser = PydanticOutputParser(pydantic_object=output_format)

Defining all the prompts that are useful for chains

In [134]:
prompt_1 = PromptTemplate.from_template(
    template = """ based on title anime {feedback} classify whether the feedback belongs to Action,Comedy,Suspence category \n {format_instruction}""",
    input_variable = ['feedback','format_instruction'],
)

Action_prompt = PromptTemplate.from_template(
    template = "suggest user with 5 {output} genre anime titles.",
    input_variable = ['output']
)
Comedy_prompt = PromptTemplate.from_template(
    template = "suggest user with 5 {output} genre anime titles.",
    input_variable = ['output']    
)
Suspence_prompt = PromptTemplate.from_template(
    template = "suggest user with 5 {output} genre anime titles.",
    input_variable = ['output']
)

defining all the chains used for chatmodel

In [135]:
classifiy_chain = prompt_1 | model | pydantic_parser  
Action_chain = Action_prompt | model | parser 
Comedy_chain = Comedy_prompt | model | parser 
Suspence_chain = Suspence_prompt | model | parser 

Creating a conditional chain based on the category it will generate different outputs

In [136]:
branch_chain = RunnableBranch(
    (lambda x : x.category == "Action" , Action_chain),
    (lambda x : x.category == "Comedy" , Comedy_chain),
    (lambda x : x.category == "Suspence", Suspence_chain),
    RunnableLambda(lambda x : "Could not find appropriate category.")
)

creating the final chain

In [141]:
chain = classifiy_chain | branch_chain 
response = chain.invoke(
    {
        "feedback" : "Grand Blue",
        "format_instruction" : pydantic_parser.get_format_instructions()
    }
)

In [142]:
print(response)

Here are **5 highly recommended Comedy anime** across different styles—from slapstick to satire—with a mix of classic and modern picks:

---

### 1. **"Gintama"** *(2006–2018)*
   - **Why?** A masterpiece of meta-humor, parody, and heartfelt storytelling. Set in an Edo-period Japan invaded by aliens, it blends absurd comedy with emotional depth.
   - **Episodes:** 367 (main series) + movies/OVAs.
   - **Watch Order:** Start with *Gintama (2006)* (skip recaps in later seasons).
   - **Bonus:** Features one of anime’s best comedy trios (Gintoki, Shinpachi, Kagura).

---

### 2. **"The Disastrous Life of Saiki K."** *(2016–2018)*
   - **Why?** A hilarious slice-of-life about a psychic teenager, Saiki, who just wants a normal life but is surrounded by eccentric characters. Short episodes (5 mins) make it easy to binge.
   - **Episodes:** 120 (total across 3 seasons) + a 2023 reboot.
   - **Style:** Dry humor, exaggerated reactions, and clever gags.

---

### 3. **"KonoSuba: God’s Blessing 